In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}


# 5-way comparison · Strategy D · D — Fabric RLM (v6-custom, reflect_only)

Wheel: `fabric_rlm-0.1.11.dev5-py3-none-any.whl` · Dataset: `longcot_cs_hard_holdout25.jsonl`

Shared `RUN_ID`: `full-20260502-110000` (joins across A/C/D/E + local B for analysis).


In [ ]:
import os, sys, json, time, traceback, uuid, platform as _plat, subprocess, re
from pathlib import Path
WHEEL_PATH = "/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.11.dev5-py3-none-any.whl"
DATASET_PATH = "/lakehouse/default/Files/fabric_rlm_longcot/datasets/longcot_cs_hard_holdout25.jsonl"
STRATEGY = "D"
STRATEGY_LABEL = "fabric_reflect"
RUN_ID = "full-20260502-110000"
SMOKE_N = None
TIER = "comparison_5way"
FILES_ROOT = Path("/lakehouse/default/Files")
RUN_ROOT = FILES_ROOT / "fabric_rlm_adaptive_validation" / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = RUN_ROOT / f"summary_{STRATEGY_LABEL}.json"
RESULTS_PATH = RUN_ROOT / f"results_{STRATEGY_LABEL}.jsonl"
TRACES_DIR = RUN_ROOT / f"traces_{STRATEGY_LABEL}"
TRACES_DIR.mkdir(parents=True, exist_ok=True)

summary = {"tier": TIER, "run_id": RUN_ID, "strategy": STRATEGY,
           "strategy_label": STRATEGY_LABEL, "started_at": time.time(),
           "wheel": WHEEL_PATH, "smoke_n": SMOKE_N,
           "python": _plat.python_version(), "stages": [], "results_summary": {}}

def write_summary():
    summary["elapsed_seconds"] = time.time() - summary["started_at"]
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str))

def stage(name, **info):
    summary["stages"].append({"stage": name, "t": round(time.time()-summary["started_at"],1), **info})
    write_summary(); print("[stage]", name, info)

stage("setup", run_root=str(RUN_ROOT))
subprocess.check_call(["pip","install","--quiet","--force-reinstall","--no-deps", WHEEL_PATH])
stage("pip_wheel", done=True)
subprocess.check_call(["pip","install","--quiet","dspy>=3.0.4"])
stage("pip_dspy", done=True)
import dspy, fabric_rlm
stage("imported", dspy=dspy.__version__, fabric_rlm=fabric_rlm.__version__)


In [ ]:
rows = []
for line in Path(DATASET_PATH).read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line: rows.append(json.loads(line))
if SMOKE_N: rows = rows[:SMOKE_N]
stage("dataset_loaded", n=len(rows))

CS_JSON_OBJECT_TEMPLATES = {"HM","MFMC","Scheduling","TM","MCM","LLVM"}
CS_INTEGER_TEMPLATES = {"VLIW","CodeTrace"}
CS_INTEGER_LIST_TEMPLATES = {"Backprop","DistMem"}
INT_RE = re.compile(r"-?\d+")
INT_CSV_RE = re.compile(r"-?\d+(?:\s*,\s*-?\d+)+")

def _resp_text(resp):
    if resp is None: return ""
    if isinstance(resp, str): return resp
    return str(resp)

def _extract_solution(text):
    if "</think>" in text:
        text = text.split("</think>", 1)[-1]
    return text.strip() or None

def _extract_last_json_object(text):
    if not text: return None
    last = None
    depth = 0; start = -1
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start >= 0:
                last = text[start:i+1]
                start = -1
    if not last: return None
    try: return json.loads(last)
    except Exception:
        try: return json.loads(last.replace("'", '"'))
        except Exception: return None

def _parse_int_list(text):
    if not text: return None
    m = INT_CSV_RE.search(text)
    if m:
        return [int(x.strip()) for x in m.group(0).split(",")]
    nums = INT_RE.findall(text)
    return [int(n) for n in nums] if nums else None

def grade(template, gold_answer, response_text):
    text = _resp_text(response_text)
    sol = _extract_solution(text) or text
    expected = gold_answer
    if isinstance(expected, str):
        try: expected = json.loads(expected)
        except Exception: pass
    if template in CS_JSON_OBJECT_TEMPLATES:
        cand = _extract_last_json_object(sol) or _extract_last_json_object(text)
        return cand == expected
    if template in CS_INTEGER_TEMPLATES:
        m = INT_RE.search(sol) or INT_RE.search(text)
        if m is None: return False
        try: return int(m.group(0)) == int(str(expected).strip())
        except Exception: return False
    if template in CS_INTEGER_LIST_TEMPLATES:
        if isinstance(expected, list):
            exp_list = [int(x) for x in expected]
        else:
            exp_list = _parse_int_list(str(expected))
        pred = _parse_int_list(sol) or _parse_int_list(text)
        return pred == exp_list
    return False

stage("validator_ready")


In [ ]:
from fabric_rlm import RLM, FabricLM
os.environ["FABRIC_RLM_CAPTURE_TURNS"] = "1"
base_lm = FabricLM("gpt-5", reasoning_effort="minimal", cache=False)
stage("lm_built", model="gpt-5", effort="minimal")


In [ ]:
os.environ["FABRIC_RLM_PVR_MODE"] = "reflect_only"
os.environ.pop("FABRIC_RLM_PVR", None)
stage("pvr_mode_set", mode=os.environ["FABRIC_RLM_PVR_MODE"])

with RESULTS_PATH.open("w", encoding="utf-8") as out_fh:
    for idx, row in enumerate(rows):
        qid = row["question_id"]; tpl = row["template"]; gold = row.get("answer")
        rec = {"strategy": STRATEGY_LABEL, "question_id": qid, "template": tpl,
               "started_at": time.time()}
        try:
            rlm = RLM(signature="question -> answer", lm=base_lm,
                      engine="v6-custom", max_turns=8)
            t0 = time.perf_counter()
            result = rlm.run({"question": row["prompt"]})
            elapsed = time.perf_counter() - t0
            ans = (result.payload or {}).get("answer") if result.payload else None
            traj = result.trajectory
            turn_records = list(getattr(traj, "turns", []) or []) if traj else []
            turns = [t.to_dict() if hasattr(t, "to_dict") else t for t in turn_records]
            prompt_tok = sum((getattr(t, "prompt_tokens", None) or 0) for t in turn_records)
            completion_tok = sum((getattr(t, "completion_tokens", None) or 0) for t in turn_records)
            passed = bool(result.submitted) and grade(tpl, gold, ans) if ans is not None else False
            rec.update({
                "passed": bool(passed), "submitted": result.submitted,
                "elapsed_seconds": elapsed,
                "prompt_tokens": prompt_tok, "completion_tokens": completion_tok,
                "n_attempts": 1, "n_turns": len(turns) if turns else None,
                "answer_preview": (str(ans)[:1000] if ans is not None else None),
            })
            trace = {"strategy": STRATEGY_LABEL, "question_id": qid, "template": tpl,
                     "prompt": row["prompt"], "answer": str(ans) if ans is not None else None,
                     "submitted": result.submitted, "passed": rec["passed"],
                     "turns": turns, "metadata": traj.metadata if traj else None}
            (TRACES_DIR / f"trace_{qid}.json").write_text(json.dumps(trace, default=str, indent=2), encoding="utf-8")
        except Exception as exc:
            rec.update({"passed": False, "error": repr(exc),
                        "traceback": traceback.format_exc()})
        out_fh.write(json.dumps(rec, default=str) + "\n"); out_fh.flush()
        stage("q_done", idx=idx+1, qid=qid, passed=rec.get("passed"),
              elapsed=round(rec.get("elapsed_seconds") or 0, 1),
              tokens=(rec.get("prompt_tokens",0) or 0)+(rec.get("completion_tokens",0) or 0))


In [ ]:
by_template = {}
all_rows = [json.loads(l) for l in RESULTS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
total = len(all_rows); passed = sum(1 for r in all_rows if r.get("passed"))
total_prompt = sum(r.get("prompt_tokens",0) or 0 for r in all_rows)
total_completion = sum(r.get("completion_tokens",0) or 0 for r in all_rows)
total_elapsed = sum(r.get("elapsed_seconds",0) or 0 for r in all_rows)
for r in all_rows:
    t = r.get("template","?")
    bt = by_template.setdefault(t, {"n":0, "passed":0})
    bt["n"] += 1
    if r.get("passed"): bt["passed"] += 1
summary["results_summary"] = {
    "n": total, "n_passed": passed,
    "pass_rate": (passed/total) if total else 0.0,
    "total_prompt_tokens": total_prompt,
    "total_completion_tokens": total_completion,
    "total_tokens": total_prompt + total_completion,
    "total_elapsed_seconds": total_elapsed,
    "by_template": by_template,
}
write_summary()
stage("done")
print(json.dumps(summary["results_summary"], indent=2, default=str))
